# 03 · From sampled derivatives to an EVT assessment

**Goal:** understand the statistical stage and its assumptions. We sample a
candidate sublevel shell, seek violations, form block maxima, fit a generalized
extreme value (GEV) distribution, and examine endpoint uncertainty.

This notebook runs independently on CPU. It uses **independent uniform rejection
samples** for the EVT illustration, rather than the research scripts' reflected
stochastic gradient Langevin dynamics (SGLD). A short adversarial search is shown
separately. This keeps the sampling mechanism visible; it is not a reproduction
of the research certification procedure or an asserted confidence guarantee.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

# Work from the repository root or anywhere inside it.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "tutorials/_support.py").is_file())
if str(ROOT / "tutorials") not in sys.path:
    sys.path.insert(0, str(ROOT / "tutorials"))
from _support import (setup, quadratic_matrix, warm_start, lie_derivative,
                      grid, sample_shell, VanDerPol)
rng = setup(42)
physics = VanDerPol(mu=1.0)
plt.rcParams.update({"figure.figsize": (6, 4), "axes.grid": False})
print("CPU tutorial | PyTorch", torch.__version__)

In [ ]:
USE_TRAINED_CHECKPOINT = False
model = warm_start()
if USE_TRAINED_CHECKPOINT:
    checkpoint = ROOT / "tutorials/outputs/tutorial_candidate.pt"
    if not checkpoint.exists():
        raise FileNotFoundError("Run notebook 02 first, or set USE_TRAINED_CHECKPOINT=False")
    saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
    model.load_state_dict(saved["state_dict"])
model.eval()
print("Candidate:", "notebook 02 trained model" if USE_TRAINED_CHECKPOINT else "independent warm start")

## 1. Define the region being assessed

We examine $D_\rho=\{x:\rho_{\rm core}\le V(x)\le\rho\}$.
Excluding the origin prevents $\dot V(0)=0$ from trivially determining the
maximum. It also leaves a **gap in a full ROA argument**: a separate local
Lyapunov/invariance argument is needed inside the core. No such proof is claimed.

Uniform rejection sampling starts in a box containing the whole sublevel.
The helper uses a positive quadratic lower bound on $V$ to choose that box and
raises an error if its sampling budget is exhausted. For $Q$ partitioned after
the first two features, completing the square gives
$V\ge x^T(Q_{11}-Q_{12}Q_{22}^{-1}Q_{21})x$.
This avoids silently restricting assessment to an arbitrary plotting window.

In [ ]:
RHO_CORE, RHO = 0.03, 0.6
SAMPLE_COUNT = 6000
samples = sample_shell(model, SAMPLE_COUNT, RHO_CORE, RHO, seed=123)
values, derivatives, _ = lie_derivative(model, samples)
raw = derivatives.detach().numpy()
assert np.isfinite(raw).all()
assert ((values >= RHO_CORE) & (values <= RHO)).all()
print(f"Observed maximum: {raw.max():.6f}; positive observations: {(raw > 0).sum()}")
fig, ax = plt.subplots()
cloud = ax.scatter(*samples[:1500].T.numpy(), c=raw[:1500], s=5, cmap="coolwarm")
fig.colorbar(cloud, ax=ax, label="Lie derivative")
ax.set(xlabel="$x_1$", ylabel="$x_2$", title="Independent samples in the shell")
plt.show()

## 2. Search actively for a counterexample

Gradient ascent on $\dot V$ seeks larger derivatives. We reject steps leaving the
shell; this elementary search can get stuck and is not the research reflected
SGLD sampler. Its optimized states are dependent and biased, so **we do not mix
them into the independent EVT sample**. A found positive derivative would reject
the candidate on this shell immediately.

In [ ]:
particles = samples[:128].clone()
for step in range(50):
    _, scores, x = lie_derivative(model, particles, create_graph=True)
    direction = torch.autograd.grad(scores.sum(), x)[0]
    proposal = (x + 0.02 * direction).detach()
    with torch.no_grad():
        trial_value = model(proposal).reshape(-1)
        inside = (trial_value >= RHO_CORE) & (trial_value <= RHO)
        particles = torch.where(inside[:, None], proposal, particles)
_, attacked, _ = lie_derivative(model, particles)
attack_max = attacked.max().item()
print(f"Search maximum: {attack_max:.6f}")
print("Counterexample found" if attack_max > 0 else "No positive derivative found by this search")

## 3. Form block maxima and fit the tail

Split the independent observations into equal blocks and take each maximum.
The GEV family is an asymptotic model for normalized maxima under suitable
conditions; a fit to finite blocks is an approximation to assess, not a theorem
about this particular dataset.

SciPy's shape parameter `c` has the opposite sign to the common EVT parameter
$\xi$. For `c > 0`, the fitted finite upper endpoint is
$e=\mathrm{loc}+\mathrm{scale}/c$; otherwise there is no finite upper endpoint.
See [SciPy's GEV convention](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.genextreme.html).
We retain infinite endpoints as inconclusive instead of discarding them.

In [ ]:
from scipy.stats import genextreme

BLOCK_SIZE = 40
maxima = raw[:len(raw)//BLOCK_SIZE*BLOCK_SIZE].reshape(-1, BLOCK_SIZE).max(axis=1)

def fit_endpoint(data):
    # Standardize for numerical conditioning, then return original units.
    center, spread = float(np.mean(data)), float(np.std(data))
    if not np.isfinite(data).all() or spread <= 1e-12:
        return None, np.inf
    try:
        c, loc, scale = genextreme.fit((data - center) / spread)
        params = (c, center + spread*loc, spread*scale)
        if not np.isfinite(params).all() or params[2] <= 0:
            return None, np.inf
        endpoint = params[1] + params[2]/c if c > 0 else np.inf
        if endpoint < np.max(data):
            return None, np.inf
        return params, endpoint
    except (ValueError, FloatingPointError, RuntimeError):
        return None, np.inf

params, endpoint = fit_endpoint(maxima)
print(f"Blocks: {len(maxima)}; fitted endpoint: {endpoint:.6f}")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(maxima, bins=18, density=True, alpha=0.6, label="Block maxima")
if params is not None:
    probabilities = (np.arange(len(maxima)) + 0.5) / len(maxima)
    theoretical = genextreme.ppf(probabilities, *params)
    axis = np.linspace(maxima.min(), maxima.max(), 200)
    axes[0].plot(axis, genextreme.pdf(axis, *params), label="Fitted GEV")
    axes[1].scatter(theoretical, np.sort(maxima), s=12)
    bounds = [min(theoretical.min(), maxima.min()), max(theoretical.max(), maxima.max())]
    axes[1].plot(bounds, bounds, "k--")
axes[0].legend()
axes[0].set(xlabel="Block maximum", ylabel="Density", title="Tail-model fit")
axes[1].set(xlabel="Fitted quantile", ylabel="Observed quantile", title="Q–Q diagnostic")
plt.tight_layout()
plt.show()

## 4. Bootstrap endpoint sensitivity

Resample the block maxima, refit, and inspect the 95th percentile of the endpoints.
We use only **40 resamples** to keep the lesson fast. This is a rough sensitivity
exercise, not a validated 95% confidence bound: endpoint estimation is delicate,
the finite-block approximation matters, and more replicates alone do not prove
coverage. Failed or unbounded fits contribute infinity. A high percentile can
therefore be infinite, which is an informative inconclusive outcome.

We also compare the proposed bound with the independent adversarial diagnostic.
If search finds a derivative above the bound, we mark the assessment inconclusive
even when all derivatives are negative. A visually good tail fit can miss a value
that optimization finds, especially in the trained-checkpoint mode.


In [ ]:
BOOTSTRAPS = 40
bootstrap_rng = np.random.default_rng(2026)
endpoints = np.array([fit_endpoint(bootstrap_rng.choice(maxima, len(maxima), replace=True))[1]
                      for _ in range(BOOTSTRAPS)])
# An order statistic avoids arithmetic interpolation between finite and infinite values.
upper = np.quantile(endpoints, 0.95, method="higher")
print(f"Finite bootstrap endpoints: {np.isfinite(endpoints).sum()}/{BOOTSTRAPS}")
print(f"Illustrative upper percentile: {upper:.6f}")
if raw.max() > 0 or attack_max > 0:
    print("REJECT on this shell: an observed point violates decrease.")
elif np.isfinite(upper) and attack_max > upper:
    print("INCONCLUSIVE: the search exceeds the bootstrap bound; the tail fit misses observed values.")
elif params is not None and endpoint < 0 and np.isfinite(upper) and upper < 0:
    print("Negative fitted tail: statistical evidence only; no ROA certificate.")
else:
    print("INCONCLUSIVE: the fitted tail does not support a negative finite endpoint.")

finite = endpoints[np.isfinite(endpoints)]
fig, ax = plt.subplots()
if len(finite):
    ax.hist(finite, bins=12, alpha=0.7)
ax.axvline(0, color="red", linestyle="--", label="Zero derivative")
ax.set(xlabel="Fitted endpoint", ylabel="Resamples", title="Bootstrap sensitivity (finite fits only)")
ax.legend()
plt.show()

## 5. Vary the block size before believing a fit

Larger blocks push maxima further into the tail but leave fewer observations to
fit. Sensitivity is a reason for more investigation. For a fixed candidate and
fixed inner core, the true maximum is nondecreasing as rho grows, but noisy fitted
endpoints need not be. Blind bisection on fitted acceptance can be misleading.

In [ ]:
for block_size in [20, 40, 80]:
    blocks = raw[:len(raw)//block_size*block_size].reshape(-1, block_size).max(axis=1)
    fitted, estimate = fit_endpoint(blocks)
    print(f"block size={block_size:3d} | blocks={len(blocks):3d} | endpoint={estimate:.6f}")

## What this says about the research method

The research scripts add reflected SGLD, particle pruning, shell searches, and
larger sampling budgets. Read `experiments/van_der_pol/EVT/verify_evt_new.py` alongside this
notebook: `compute_V_Vdot` computes the same chain rule; `volume_attack_batch`
changes sampling; `analyze_evt_block_maxima` fits the tail. Particle dependence,
coverage, fitted-model diagnostics, and the excluded core all need justification
before turning this evidence into a statistical guarantee. A usual KS p-value
with parameters fitted to the same data is not a calibrated goodness-of-fit test.

**Try it:**

1. Run notebook 02, enable `USE_TRAINED_CHECKPOINT`, and compare candidate geometry
   and endpoint sensitivity. Keep the candidate fixed during each assessment.
2. Increase `RHO` until an actual positive derivative is found. Locate that point
   on a contour plot. Which is stronger evidence: that point or a favorable fit?
3. Change the seed and block size; then increase the bootstrap budget. Separate
   Monte Carlo variability from errors in the tail-model assumptions.
4. Read `experiments/dim_scaling/EVT/lyapunov_core_new.py`. Why does uniform rejection become
   expensive in high dimension? What does SGLD gain, and which assumptions change?

See [experiment notes](../docs/experiments.md) for the baseline implementations
and known limitations. None of these notebook diagnostics proves global stability.